# 🚇 TunnelFlow-UB — Colab + free T4 → public Ollama in one click

### GPU (1 time, 3 clicks)
> **Runtime → Change runtime type → Hardware accelerator: `T4 GPU` → Save**

Colab has no API/hook to switch the runtime from code (the `accelerator` metadata in this file is set to `GPU`/`T4`,
but free Colab does not read it — Kaggle and Colab Enterprise do). So cell 1 checks the real hardware:
with a T4 it runs `tranhuythang9999/deepseek-v4-pro-9b-heretic:probe`; without one it warns and falls back to a small CPU model, so Run all never dead-ends.

### Then
**Connect** and press ▶ on cell 1 (or **Run all**). That one cell installs Bore + Ollama, starts the server, opens the
tunnel, **verifies the public port with `curl`**, pulls the model and calls it through the public URL — the port it prints
is already proven to work.

> Colab cannot run code on *Connect* either; one press of ▶ on cell 1 is the minimum it allows, and it does the whole setup.

### What each cell is for

| Cell | What it does |
|---|---|
| 1 | GPU check → install Bore + Ollama → start server → open tunnel → verify port → pull model → call model through the public URL |
| 2 | (optional) SSH into Colab — skipped automatically until the cell 1 tunnel is healthy |
| 3 | Copy-paste snippets for your own machine (curl / Python / OpenAI SDK) |
| 4 | Runnable Python client (OpenAI SDK, stream + non-stream) |
| 5 | Runnable curl client (native API + OpenAI-compatible API) |

### The bug this repo fixes
```text
real bore log : 2026-11-12T09:41:07.123Z  INFO bore_cli::client: listening at bore.pub:41234
old code      : grep -o '[0-9]\{4,5\}' | head -n 1               -> 2026    (grabbed the YEAR)
new code      : grep -oE 'bore\.pub:[0-9]{1,5}' | sed 's/.*://'  -> 41234   (correct)
```
Plus: wait loop instead of `sleep 4`, ANSI/`\r` stripped, `set -e` safe, the port is verified with `curl` before it is
printed, and tunnel cleanup is per-port so the SSH cell never kills the Ollama tunnel.

> ⚠️ A free Colab runtime stops after ~12 h or when idle → re-run cell 1, the port changes.


In [ ]:
%%bash
# ==========================================================
# CELL 1 — everything: GPU check, Ollama, tunnel, model, test
# Safe to re-run: it is idempotent.
# ==========================================================
set -e

# ---------------- CONFIG ----------------
LOCAL_PORT=11434
REMOTE_HOST="bore.pub"
MODEL="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
CPU_MODEL="qwen2.5:0.5b"   # fallback when the runtime has no GPU
# ----------------------------------------

echo "=================================================="
echo "0/6. GPU check (free Colab T4 gives full speed)..."
echo "=================================================="
if command -v nvidia-smi > /dev/null 2>&1 && nvidia-smi -L > /dev/null 2>&1; then
    echo "GPU: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -n 1)"
else
    echo "WARN: no GPU on this runtime."
    echo "      For full speed: Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again."
    echo "      Continuing with the small CPU model instead: ${CPU_MODEL}"
    MODEL="${CPU_MODEL}"
fi

echo "=================================================="
echo "1/6. Installing Bore (TCP tunnel)..."
echo "=================================================="
if command -v bore > /dev/null 2>&1; then
    echo "OK: $(bore --version)"
else
    wget -q https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    tar -xzf bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    mv bore /usr/local/bin/
    rm -f bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    echo "OK: $(bore --version)"
fi

echo "=================================================="
echo "2/6. Installing dependencies and Ollama..."
echo "=================================================="
apt-get update -qq
apt-get install -y -qq zstd curl ca-certificates > /dev/null
if command -v ollama > /dev/null 2>&1; then
    echo "OK: ollama already installed: $(ollama --version)"
else
    # install.sh often fails at the systemctl step (Colab has no systemd) -> do not abort
    curl -fsSL https://ollama.com/install.sh | sh || echo "WARN: install.sh reported an error (usually missing systemd); checking the binary next."
fi
if ! command -v ollama > /dev/null 2>&1; then
    echo "ERROR: could not install Ollama."
    exit 1
fi

echo "=================================================="
echo "3/6. Starting Ollama on 0.0.0.0:${LOCAL_PORT}..."
echo "=================================================="
# Only this port's tunnel is killed, never the SSH tunnel
pkill -f "ollama serve" || true
pkill -f "bore local ${LOCAL_PORT} --to" || true
sleep 2

nohup env OLLAMA_HOST=0.0.0.0:${LOCAL_PORT} OLLAMA_ORIGINS="*" ollama serve > ollama.log 2>&1 &

OLLAMA_OK=0
for _ in $(seq 1 60); do
    if curl -sf "http://127.0.0.1:${LOCAL_PORT}/api/tags" > /dev/null 2>&1; then
        OLLAMA_OK=1
        break
    fi
    sleep 1
done
if [ "$OLLAMA_OK" != "1" ]; then
    echo "ERROR: Ollama is not responding. Last log lines:"
    tail -n 30 ollama.log
    exit 1
fi
echo "OK: Ollama is up locally"

echo "=================================================="
echo "4/6. Opening the tunnel and reading the CORRECT port..."
echo "=================================================="
rm -f bore.log
nohup bore local ${LOCAL_PORT} --to ${REMOTE_HOST} > bore.log 2>&1 &
BORE_PID=$!

# OLD BUG: grep -o '[0-9]{4,5}' on the line
#   "2026-11-12T09:41:07.123Z  INFO bore_cli::client: listening at bore.pub:41234"
#   matched the YEAR "2026" first => printed http://bore.pub:2026 (wrong)
# FIX: only the digits right after "bore.pub:"
PORT=""
for _ in $(seq 1 30); do
    PORT=$(sed 's/\x1b\[[0-9;]*m//g' bore.log 2>/dev/null \
           | tr -d '\r' \
           | grep -oE "bore\.pub:[0-9]{1,5}" \
           | sed 's/.*://' \
           | tail -n 1) || true
    if [ -n "$PORT" ]; then
        break
    fi
    sleep 1
done
if [ -z "$PORT" ]; then
    echo "ERROR: could not read the port. Raw bore log:"
    sed 's/\x1b\[[0-9;]*m//g' bore.log | tail -n 20
    kill -0 "$BORE_PID" 2>/dev/null || echo "The bore process already exited."
    exit 1
fi
echo "$PORT" > bore.port
BASE="http://${REMOTE_HOST}:${PORT}"
echo "Public port: $PORT"

TUNNEL_OK=0
for _ in $(seq 1 15); do
    if curl -sf --max-time 10 "${BASE}/api/tags" > /dev/null 2>&1; then
        TUNNEL_OK=1
        break
    fi
    sleep 2
done
if [ "$TUNNEL_OK" = "1" ]; then
    echo "OK: tunnel verified over the internet -> $BASE"
else
    echo "WARN: port parsed but the public check is not passing yet (bore.pub can be slow)."
fi

echo "=================================================="
echo "5/6. Pulling $MODEL (first run only, several minutes)..."
echo "=================================================="
if ! ollama pull "$MODEL"; then
    echo "ERROR: pull failed for $MODEL"
    echo "Hint: if the model is private on ollama.com, make it public so Colab can pull it."
    exit 1
fi

echo "=================================================="
echo "6/6. Calling the model THROUGH THE PUBLIC URL..."
echo "=================================================="
curl -s --max-time 300 "${BASE}/api/generate" \
    -d "{\"model\":\"$MODEL\",\"prompt\":\"Answer in one short sentence: what is 1+1?\",\"stream\":false}" \
    | python3 -c "import sys,json;d=json.load(sys.stdin);print('Model says:',(d.get('response') or '(empty)').strip())" \
    || echo "WARN: the public test call failed, try the same URL again in a few seconds."

echo "=================================================="
echo "READY"
echo "API base : $BASE"
echo "Test     : curl $BASE/api/tags"
echo "Python   : base_url='$BASE/v1'"
echo "Model    : $MODEL"
echo "=================================================="


In [ ]:
%%bash
# ==========================================================
# CELL 2 (optional) — SSH into Colab.
# Runs ONLY when the cell 1 tunnel is already healthy.
# Change ROOT_PASSWORD before using this.
# ==========================================================
ENABLE_SSH=1
ROOT_PASSWORD="${ROOT_PASSWORD:-MySecurePass123}"

if [ "$ENABLE_SSH" != "1" ]; then
    echo "SKIP: ENABLE_SSH=0."
    exit 0
fi

PORT_OLLAMA=$(cat bore.port 2>/dev/null || true)
if [ -z "$PORT_OLLAMA" ] || ! curl -sf --max-time 10 "http://bore.pub:${PORT_OLLAMA}/api/tags" > /dev/null 2>&1; then
    echo "SKIP: the Ollama tunnel is not healthy yet. Run Cell 1 first."
    exit 0
fi
echo "OK: Ollama tunnel is healthy (port ${PORT_OLLAMA}) -> setting up SSH."

echo "=================================================="
echo "1. Installing the SSH server..."
echo "=================================================="
apt-get install -y -qq openssh-server > /dev/null 2>&1 || true

echo "=================================================="
echo "2. Setting the root password (non-interactive)..."
echo "=================================================="
echo "root:${ROOT_PASSWORD}" | chpasswd

echo "=================================================="
echo "3. Configuring sshd for a container environment..."
echo "=================================================="
mkdir -p /run/sshd
sed -i 's/#\?PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
sed -i 's/#\?PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
ssh-keygen -A > /dev/null 2>&1
grep -E "PermitRootLogin|PasswordAuthentication" /etc/ssh/sshd_config

echo "=================================================="
echo "4. Starting sshd..."
echo "=================================================="
pkill sshd || true
sleep 1
/usr/sbin/sshd
ss -tlnp 2>/dev/null | grep sshd || echo "WARN: sshd is not listening yet."

echo "=================================================="
echo "5. Opening a Bore tunnel for port 22..."
echo "=================================================="
pkill -f "bore local 22 --to" 2>/dev/null || true   # only the old SSH tunnel, keep the Ollama one
rm -f bore_ssh.log
nohup bore local 22 --to bore.pub > bore_ssh.log 2>&1 &

SSH_PORT=""
for _ in $(seq 1 30); do
    SSH_PORT=$(sed 's/\x1b\[[0-9;]*m//g' bore_ssh.log 2>/dev/null \
               | tr -d '\r' \
               | grep -oE "bore\.pub:[0-9]{1,5}" \
               | sed 's/.*://' \
               | tail -n 1) || true
    if [ -n "$SSH_PORT" ]; then
        break
    fi
    sleep 1
done

if [ -n "$SSH_PORT" ]; then
    echo "DONE: SSH public port: $SSH_PORT"
    echo "ssh -p ${SSH_PORT} root@bore.pub"
    echo "Password: ${ROOT_PASSWORD}"
else
    echo "WARN: could not read the SSH port. Raw log:"
    sed 's/\x1b\[[0-9;]*m//g' bore_ssh.log | tail -n 20
fi


## Snippets for your own machine

Use the port printed by cell 1 (it is also stored in `bore.port`). Replace `<PORT>`.

```bash
# health check
curl http://bore.pub:<PORT>/api/tags

# one-shot prompt (native Ollama API)
curl http://bore.pub:<PORT>/api/generate \
  -d '{"model":"tranhuythang9999/deepseek-v4-pro-9b-heretic:probe","prompt":"hello","stream":false}'
```

```python
# OpenAI-compatible API
# pip install openai
from openai import OpenAI

client = OpenAI(base_url="http://bore.pub:<PORT>/v1", api_key="ollama")
print(client.chat.completions.create(
    model="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe",
    messages=[{"role": "user", "content": "Hello!"}],
).choices[0].message.content)
```

```python
# streaming
stream = client.chat.completions.create(
    model="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe",
    messages=[{"role": "user", "content": "Tell me a 3-line story."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)
```

```python
# no SDK, plain requests
import requests

r = requests.post(
    "http://bore.pub:<PORT>/api/generate",
    json={"model": "tranhuythang9999/deepseek-v4-pro-9b-heretic:probe", "prompt": "Hi", "stream": False},
    timeout=300,
)
print(r.json()["response"])
```

```bash
# SSH into Colab (port printed by cell 2)
ssh -p <SSH_PORT> root@bore.pub

# inspect / stop everything
pgrep -a ollama; pgrep -a bore
pkill ollama; pkill bore
```


In [ ]:
# ==========================================================
# CELL 4 — Runnable Python client (OpenAI SDK), stream + non-stream
# ==========================================================
import time
from pathlib import Path

PORT_FILE = Path("bore.port")
if not PORT_FILE.exists():
    raise SystemExit("Run cell 1 first: there is no bore.port yet.")

MODEL = "tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
BASE_URL = "http://bore.pub:" + PORT_FILE.read_text().strip()
print("Endpoint:", BASE_URL)
print("Model   :", MODEL)

# Ollama ignores the key, but the SDK requires one
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=BASE_URL + "/v1", api_key="ollama", timeout=300)


def ask(prompt, system="You are a concise assistant. Answer in 1-2 sentences."):
    """Blocking request."""
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
    )
    text = (resp.choices[0].message.content or "").strip()
    print(f"[{time.time() - t0:.1f}s] {text}")
    return text


def ask_stream(prompt, system="You are a helpful assistant."):
    """Streaming request: tokens printed as they arrive."""
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
        stream=True,
    )
    for chunk in stream:
        print(chunk.choices[0].delta.content or "", end="", flush=True)
    print()


try:
    ask("What is 1+1?")
    print("\\n--- streaming ---")
    ask_stream("Write 3 short lines about the ocean.")
except Exception as exc:  # noqa: BLE001
    print("Call failed:", type(exc).__name__, exc)
    print("Hint: re-run cell 1 (the tunnel or the port may have changed).")


In [ ]:
%%bash
# ==========================================================
# CELL 5 — Runnable curl client (copy-paste friendly)
# ==========================================================
PORT=$(cat bore.port 2>/dev/null || true)
if [ -z "$PORT" ]; then
    echo "SKIP: no bore.port yet. Run cell 1 first."
    exit 0
fi
MODEL="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
BASE="http://bore.pub:${PORT}"
echo "Base URL: ${BASE}"

echo
echo "1) GET /api/tags (models available)"
curl -s --max-time 30 "${BASE}/api/tags" | head -c 400
echo

echo
echo "2) POST /api/generate (native Ollama API)"
curl -s --max-time 300 "${BASE}/api/generate" \
    -d "{\"model\":\"${MODEL}\",\"prompt\":\"Hello in 5 words\",\"stream\":false}" \
    | python3 -c "import sys,json;d=json.load(sys.stdin);print((d.get('response') or '(empty)').strip())" \
    || echo "WARN: /api/generate failed"

echo
echo "3) POST /v1/chat/completions (OpenAI-compatible API)"
curl -s --max-time 300 "${BASE}/v1/chat/completions" \
    -H "Content-Type: application/json" \
    -d "{\"model\":\"${MODEL}\",\"messages\":[{\"role\":\"user\",\"content\":\"Say hi in 5 words\"}]}" \
    | python3 -c "import sys,json;d=json.load(sys.stdin);print(d['choices'][0]['message']['content'].strip())" \
    || echo "WARN: /v1/chat/completions failed"
